# SolarSentinel, Milestone 4: AI component and experiments

Plan: `docs/3-milestone4-plan.md`. Every number below comes from code in this notebook or from the run records in
`experiments/results/` (`m4_*.json` + `_val_preds.parquet`). All evaluation uses the gap-corrected 2015 validation set.
Replay (2016-17) labels are never loaded (`dataset.load_observations` refuses them); section 11 compares replay
*inputs* only.

Training runs are made by `python -m solarsentinel.models.m4` (this notebook reads their records, so executing it
does not retrain). `TAG` selects a run set: `None` = the final runs, `"dryrun"` = the test run on the first 1,465
cutouts.

## 1. Setup

In [ ]:
import json, os, platform
import numpy as np, pandas as pd
import torch, timm, sklearn
from IPython.display import Image, display
from solarsentinel import storage
from solarsentinel.config import load_config, load_experiments
from solarsentinel.data import v2, cutouts_v2
from solarsentinel.models import m4, dataset, errors, metrics, m4_figures as figs

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 70)
TAG = os.environ.get("M4_TAG") or None
CFG, V2 = load_experiments()["m4"], load_config()["v2"]
print(json.dumps({"code": v2.code_version(), "python": platform.python_version(), "torch": torch.__version__,
                  "timm": timm.__version__, "sklearn": sklearn.__version__, "tag": TAG}, indent=1))
print(json.dumps({"v2_rules": V2, "m4_recipe": CFG["recipe"], "m4_eval": CFG["eval"]}, indent=1))

## 2. v2 data and split audit (feedback item 1)

Gap rule: a later block drops every observation whose `slot_time` is earlier than the previous block's last
`slot_time` + 27 days. `processed/v2/observations.parquet` is v1 filtered by this rule; `split_audit.json` holds the
counts below and a fingerprint, and the loader checks every read against it.

In [ ]:
audit = json.loads(storage.local_path(v2.audit_rel()).read_text())
display(pd.DataFrame(audit["counts"]).T)
print("first allowed slot:", audit["first_allowed_slot"])
print("HARPs shared across splits:", audit["harps_shared_across_splits"])
print("achieved minimum gap (days):", audit["achieved_min_gap_days"])
print("removed by the gap:", audit["removed_by_gap"])
print("fingerprint:", json.dumps(audit["fingerprint"], indent=1))

**First-exploration numbers before and after the gap.** No retraining: each saved `fme_*_val_preds.parquet` is
filtered to the val rows that survive the gap; the max-TSS threshold is re-picked on those cases (same convention as
the original runs).

In [ ]:
FME = m4.rescore_fme()
FME.round(3)

## 3. Frozen label rule and counts (feedback item 2)

| Use | Rule |
|---|---|
| Training | exclude flagged negatives (`has_unattributed_mx & label == 0`); keep `quality_nonzero` rows; pilot-week negatives excluded (developer decision 2026-11-02) |
| Evaluation, primary | every gap-corrected val case that has an image |
| Evaluation, secondary ("clean") | primary minus flagged negatives and `quality_nonzero` rows |

In [ ]:
_, META = dataset.load_cutouts(cutouts_v2.pad_prefix(), version="v2")
(_, MTR), (_, MVA) = m4.load_arm("pad")
OBS = dataset.load_observations("v2")

def counts(m):
    return {"cases": len(m), "positives": int(m.label.sum()), "negatives": int((m.label == 0).sum()),
            "flagged_negatives": int((m.has_unattributed_mx & (m.label == 0)).sum()),
            "quality_nonzero": int(m.quality_nonzero.sum()), "harps": m.HARPNUM.nunique(),
            "flaring_harps": m.HARPNUM[m.label == 1].nunique()}

tr_disk = META[META.split == "train"]
LABEL_COUNTS = pd.DataFrame({
    "train, all v2 rows": counts(OBS[OBS.split == "train"]),
    "train, images on disk": counts(tr_disk),
    "train, used (frozen rule)": counts(MTR),
    "val, all v2 rows": counts(OBS[OBS.split == "val"]),
    "val primary (cases with image)": counts(MVA),
    "val clean": counts(MVA[dataset.clean_mask(MVA)]),
}).T
display(LABEL_COUNTS)
print("train images from the pilot week:", int((tr_disk.fits_root == "pilot").sum()),
      "| pilot negatives excluded from training:", int(((tr_disk.fits_root == "pilot") & (tr_disk.label == 0)).sum()))

**What the clean subset leaves out (Should).** Flagged vs clean negatives over all train+val v2 rows, on three SHARP
inputs and on the share in high-activity months (months in the top quartile of positive rate, train+val).

In [ ]:
neg = OBS[OBS.label == 0].copy()
month = OBS.slot_time.dt.tz_localize(None).dt.to_period("M")
rate = OBS.groupby(month).label.mean()
high = set(rate[rate >= rate.quantile(0.75)].index)
neg["high_activity_month"] = month[neg.index].isin(high)
FLAG_CMP = neg.groupby(neg.has_unattributed_mx.rename("flagged")).agg(
    n=("HARPNUM", "size"), USFLUX_median=("USFLUX", "median"), AREA_ACR_median=("AREA_ACR", "median"),
    TOTUSJH_median=("TOTUSJH", "median"), share_in_high_activity_months=("high_activity_month", "mean"))
display(FLAG_CMP)
print("high-activity months:", sorted(str(m) for m in high))

## 4. Padding inspection (feedback item 3)

v2 rule: zero-pad the FITS array to a square (centered), then the v1 NaN fill and anti-aliased resize to 128x128
float16. Inspected: the 10 thinnest regions (largest long/short side ratio, one snapshot per HARP) and every cutout
with `nan_fraction > 0`. The figure shows 6 thin regions and the 3 HARPs with the most NaN pixels.

In [ ]:
IDX_PAD = pd.read_parquet(storage.local_path(f"{cutouts_v2.pad_prefix()}/index.parquet"))
INSP = cutouts_v2.inspect(IDX_PAD, figs.fig_dir() / "fig2_pad_vs_stretch.png")
display(pd.DataFrame(INSP.pop("thinnest")))
print(json.dumps(INSP, indent=1))
print("cutouts per FITS root:", IDX_PAD.fits_root.value_counts().to_dict(), "| pad_fraction median",
      round(float(IDX_PAD.pad_fraction.median()), 3))
MAN_PATH = storage.local_path(f"{v2.v2_prefix()}/manifest.json")
MAN = json.loads(MAN_PATH.read_text())
print("manifest:", f"{v2.v2_prefix()}/manifest.json", "sha256", storage.sha256(MAN_PATH), "| files", len(MAN["files"]),
      "| commit", MAN["git_commit"], "dirty", MAN["dirty"])
display(Image(str(figs.fig_dir() / "fig2_pad_vs_stretch.png"), width=420))

## 5. Baseline: persistence (PR1)

"Flare" at t if the same HARP had an attributed flare >= M1.0 peaking in (t - 24 h, t]: the label logic mirrored
backwards (same links, same science flux scale). It needs no images, so it also shows that the parameter-only path
works. `check_mirror` rebuilds the forward labels with the same code: it must reproduce `label` exactly.

In [ ]:
from solarsentinel.models import baseline
print("forward-label mismatches when the same code is run forwards (train+val):", baseline.check_mirror(OBS))
R2 = m4.e2(TAG)
CASES_KEYS = m4.load_run(R2["run_ids"]["pad"][0])[1] if R2["run_ids"]["pad"] else None
PERS = m4.persistence_rows(CASES_KEYS)
PERS_T = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != "bootstrap"} | {"tss_ci95": v["bootstrap"]["tss_ci95"]}
                       for k, v in PERS.items()}).T
PERS_T

## Decision rules (pre-registered before the final runs)

Shared recipe: v2 data and gap, frozen label rule, 20 epochs, final epoch evaluated (no early stopping), flips +
polarity flip, `pos_weight = n_neg/n_pos`, AdamW lr 1e-4 wd 0.05, cosine with 1 warmup epoch, batch 32, fp32 on
`mps`, seeds 42, 0, 1. Input `clip(Br, ±c)/c` with `c` = 99.5th percentile of |Br| over the arm's non-zero train
pixels (developer decision 2026-11-02, so padding zeros do not lower `c`). Every run is scored at its own max-TSS
threshold on val.

- **E2 (pad vs stretch, ResNet-18).** Adopt pad unless clearly worse: mean-over-seeds TSS difference (pad -
  stretch) < -0.02 **and** its 95% HARP-bootstrap CI excludes 0. Also reported: ROC-AUC, TSS and recall on thin
  regions (long/short side >= 2).
- **E1 (ResNet-18 vs DeiT-Tiny, on the E2 winner).** If the 95% CI of the mean TSS difference excludes 0, the
  higher TSS wins. Otherwise, in order: (1) E3 calibration result (the model whose seed-42 scores reach probability
  mode), (2) size and CPU latency against Section 2.3 (< 50 MB, < 100 ms per region), (3) implementation complexity
  (ResNet-18: plain CNN, any input size; DeiT-Tiny needs a resized patch grid). The rule that decided is recorded.
- **E3 (calibration of the selected model, seed 42).** None vs Platt vs isotonic, HARP-grouped 2-fold
  cross-fitting within val, prevalence weights. Probability mode if a calibration method passes all three PR2
  checks (mean p / observed in [0.5, 2]; calibration-slope 95% CI contains 1; 5 equal-count bins, at most one bin
  whose observed-frequency CI misses its mean p); the passing method with the higher weighted BSS is used. Else
  ranking mode.
- **Confidently wrong** (error analysis): calibrated p > 0.9 for a false positive, < 0.05 for a false negative.

## 6. E2: does keeping the regions' shape (pad) beat stretching?

In [ ]:
COLS = ["run_id", "seed", "tss", "tss_ci95", "roc_auc", "pr_auc_corrected", "threshold", "pod", "pofd", "thin_n",
        "thin_pos", "thin_tss", "thin_recall", "clip_c", "n_train", "n_train_pos", "n_val", "n_val_pos", "epoch_s_median"]

def show(name, s):
    print(name, {k: [round(v[0], 3), round(v[1], 3)] for k, v in s["mean_sd"].items()}, "(mean, sd over seeds)")
    display(s["runs"][COLS].round(3))

for arm in ("pad", "stretch"):
    if arm in R2:
        show(f"ResNet-18 {arm}", R2[arm])
print(json.dumps(R2.get("diff_pad_minus_stretch"), indent=1))
print("E2 winner:", R2.get("winner"), "| rule:", R2.get("rule"))

## 7. E1: which pretrained model, and does it beat persistence?

In [ ]:
WIN = R2.get("winner", "pad")
E1 = m4.e1(WIN, TAG)
for k, name in (("A", "ResNet-18"), ("B", "DeiT-Tiny")):
    if k in E1:
        show(f"{name} {WIN}", E1[k])
print(json.dumps({k: E1.get(k) for k in ("diff_deit_minus_resnet", "tiebreak_calibration_pass",
                                          "tiebreak_size_latency_meets_targets", "selected", "decided_by")}, indent=1, default=str))
SM = {k: m4.system_measures(k) for k in ("A", "B")}
print(json.dumps(SM, indent=1))
ARMS_FIG = {f"ResNet-18\nstretch": R2["stretch"]["runs"]} if "stretch" in R2 else {}
if "pad" in R2:
    ARMS_FIG["ResNet-18\npad"] = R2["pad"]["runs"]
if "B" in E1:
    ARMS_FIG[f"DeiT-Tiny\n{WIN}"] = E1["B"]["runs"]
display(Image(str(figs.experiments(ARMS_FIG, PERS["model_cases"])), width=380))

## 8. E3: can the selected model's scores be shown as probabilities?

In [ ]:
SEL = E1.get("selected", "A")
SEL_RID = m4.run_id("E2" if SEL == "A" else "E1", SEL, WIN, CFG["recipe"]["seed"], TAG)
E3 = m4.e3(SEL_RID)
print("selected run:", SEL_RID, "| n", E3["n"], "positives", E3["n_pos"], "| weights", E3["weights"])
E3_T = pd.DataFrame({m: {k: v for k, v in r.items() if k != "bins"} for m, r in E3["methods"].items()}).T
display(E3_T[["mean_p", "observed_rate", "ratio", "check1_ratio_pass", "slope", "slope_ci95", "check2_slope_pass",
              "bin_misses", "check3_bins_pass", "all_pass", "brier", "bss", "threshold", "tss_at_threshold", "roc_auc"]])
for m, r in E3["methods"].items():
    print(m); display(pd.DataFrame(r["bins"]).round(4))
print("dashboard mode:", E3["mode"], "| method:", E3["chosen_method"], "|", E3["rule"])
_, SEL_PREDS = m4.load_run(SEL_RID)
from solarsentinel.models import calibration as cal
W = np.where(SEL_PREDS.label.to_numpy() == 1, E3["weights"]["positive"], E3["weights"]["negative"])
curves = {"raw": SEL_PREDS.p.to_numpy().astype(float), "Platt (cross-fitted)": E3["p_cal"]["platt"],
          "isotonic (cross-fitted)": E3["p_cal"]["isotonic"]}
display(Image(str(figs.reliability(SEL_PREDS.label.to_numpy(), W, curves, CFG["eval"]["reliability_bins"])), width=520))

## 9. Error analysis (selected model, seed 42, at its threshold)

In [ ]:
SEL_REC, _ = m4.load_run(SEL_RID)
THR = SEL_REC["metrics"]["threshold"]
PCAL = E3["p_cal"][E3["chosen_method"]] if E3["chosen_method"] else None  # ranking mode: raw scores only
IDX_SEL = pd.read_parquet(storage.local_path(f"{m4.arm_prefix(WIN)}/index.parquet"))
T = errors.tag(SEL_PREDS, IDX_SEL, THR, thin_aspect=CFG["eval"]["thin_aspect"], p_cal=PCAL)
print("threshold", round(THR, 4), "| outcomes", T.outcome.value_counts().to_dict(),
      "| p_cal from", E3["chosen_method"] or "none (raw scores used)")
CC = errors.category_counts(T)
display(CC)
cand = CC[CC.category != "persistence_agrees"].sort_values("n", ascending=False).iloc[0]
LARGEST = (cand.error, cand.category)
print("largest category:", LARGEST)
CASES = errors.inspect_cases(T, LARGEST)
CASE_COLS = ["why", "outcome", "p", "p_cal", "HARPNUM", "NOAA_AR", "slot_time", "label_class", "has_unattributed_mx",
             "multi_noaa", "quality_nonzero", "USFLUX", "aspect", "nan_fraction", "LON_FWT", "persistence"]
display(CASES[CASE_COLS].round(3))
CW = errors.confidently_wrong(T, "p_cal" if PCAL is not None else "p")
print("confidently wrong:", len(CW), "| FP > 0.9:", int((CW.outcome == "FP").sum()), "| FN < 0.05:", int((CW.outcome == "FN").sum()),
      "| all FNs:", int((T.outcome == "FN").sum()))
display(CW[CASE_COLS[1:]].round(3))
PAT = errors.pattern(T)
print(json.dumps(PAT, indent=1))
display(Image(str(figs.error_gallery(CASES, m4.arm_prefix(WIN))), width=520))
display(Image(str(figs.error_pattern(PAT)), width=520))

**Task-level view and the clean subset.**

In [ ]:
TV = m4.task_view(SEL_PREDS, THR)
print("operator view at the chosen threshold, natural 2015 prevalence:", json.dumps(TV, indent=1))
cm = dataset.clean_mask(MVA)
y, p, h = SEL_PREDS.label.to_numpy(), SEL_PREDS.p.to_numpy(), SEL_PREDS.HARPNUM.to_numpy()
assert (SEL_PREDS[["HARPNUM", "T_REC"]].values == MVA[["HARPNUM", "T_REC"]].astype({"T_REC": str}).values).all()
w = metrics.neg_weights(y, CFG["eval"]["val_negatives_total"])
prim = metrics.evaluate(y, p, h, w, threshold=THR)
clean = metrics.evaluate(y[cm], p[cm], h[cm], w[cm], threshold=THR)
print("TSS primary", round(prim["tss"], 3), f"(n={prim['n']}, pos={prim['n_pos']})", "| clean", round(clean["tss"], 3),
      f"(n={clean['n']}, pos={clean['n_pos']})", "| clean - primary", round(clean["tss"] - prim["tss"], 3))

## 10. System measures (vs Section 2.3/2.6: < 50 MB, milliseconds per region on CPU, < 10 min per epoch)

In [ ]:
rows = []
RUNS_OF = {"A": R2.get(WIN, {}).get("runs"), "B": E1.get("B", {}).get("runs")}
for k, s in SM.items():
    runs = RUNS_OF[k]
    rows.append({"model": s["model"], "params": s["params"], "state_dict_mb": s["state_dict_mb"],
                 **{f"cpu_b{x['batch']}_median_ms": x["median_ms"] for x in s["latency"]},
                 **{f"cpu_b{x['batch']}_per_region_ms": x["per_region_ms"] for x in s["latency"]},
                 "threads": s["latency"][0]["threads"],
                 "s_per_epoch_mps_median": float(runs.epoch_s_median.median()) if runs is not None else None,
                 "mps_driver_mb_max": float(runs.mps_driver_mb_max.max()) if runs is not None else None})
SYS = pd.DataFrame(rows)
display(SYS)
print("tonight's export:", json.dumps(m4.export_throughput(), indent=1))
print("cost: $0 (laptop, public JSOC export, bucket reads)")

## 11. Summary tables

In [ ]:
def row(name, s, diff=None):
    ms = s["mean_sd"]
    r = s["runs"]
    return {"arm": name, "seeds": len(r), "tss_mean": ms["tss"][0], "tss_sd": ms["tss"][1], "roc_auc_mean": ms["roc_auc"][0],
            "pr_auc_corrected_mean": ms["pr_auc_corrected"][0], "thin_tss_mean": ms["thin_tss"][0],
            "thin_recall_mean": ms["thin_recall"][0], "s_per_epoch": ms["epoch_s_median"][0],
            "paired_diff": None if diff is None else diff["diff_mean_tss"], "paired_diff_ci95": None if diff is None else diff["diff_ci95"],
            "n_val": int(r.n_val.iloc[0]), "n_val_pos": int(r.n_val_pos.iloc[0])}

RES = []
if "stretch" in R2: RES.append(row("E2 ResNet-18 stretch", R2["stretch"]))
if "pad" in R2: RES.append(row("E2 ResNet-18 pad", R2["pad"], R2.get("diff_pad_minus_stretch")))
if "B" in E1: RES.append(row(f"E1 DeiT-Tiny {WIN}", E1["B"], E1.get("diff_deit_minus_resnet")))
for k, v in PERS.items():
    RES.append({"arm": f"persistence ({k})", "tss_mean": v["tss"], "paired_diff_ci95": v["bootstrap"]["tss_ci95"], "n_val": v["n"], "n_val_pos": v["n_pos"]})
RESULTS = pd.DataFrame(RES)
display(RESULTS.round(3))

conf = []
for rid in [*R2["run_ids"].get("pad", []), *R2["run_ids"].get("stretch", []), *E1.get("run_ids", {}).get("B", [])]:
    rec, _ = m4.load_run(rid)
    x = rec.get("extra", {})
    conf.append({"run_id": rid, "model": rec["model"]["name"], "hf_repo": rec["model"]["hf_repo"], "timm": rec["versions"]["timm"],
                 "torch": rec["versions"]["torch"], "dataset": x.get("dataset_version"), "manifest_sha256": (x.get("manifest_sha256") or "")[:12],
                 "seed": rec["seed"], "epochs": rec["recipe"]["epochs"], "clip_c": round(rec["clip_c"], 1),
                 "n_train": rec["data"]["n_train"], "n_train_pos": rec["data"]["n_train_pos"], "n_val": rec["data"]["n_val"],
                 "n_val_pos": rec["data"]["n_val_pos"], "git_commit": (x.get("git_commit") or "")[:10], "dirty": x.get("dirty")})
CONFIG = pd.DataFrame(conf)
display(CONFIG)

**R9 input comparison (Should; replay inputs only, no labels).** `quality_nonzero` vs clean replay records on the 18
SHARP parameters (median, % difference), quality flags per month and distinct HARPs. Replay images are not downloaded,
so image statistics are not available here.

In [ ]:
KEYS = load_config()["sources"]["sharp"]["keys"]["summary"]
RP = dataset.load_observations("v2", ("replay",), columns=["HARPNUM", "slot_time", "quality_nonzero", "QUALITY", *KEYS])
med = RP.groupby("quality_nonzero")[KEYS].median().T
med.columns = ["clean_median", "quality_nonzero_median"]
med["pct_difference"] = (100 * (med.quality_nonzero_median - med.clean_median) / med.clean_median.abs()).round(1)
display(med)
mon = RP.assign(month=RP.slot_time.dt.tz_localize(None).dt.to_period("M")).groupby("month").agg(
    records=("HARPNUM", "size"), quality_nonzero=("quality_nonzero", "sum"), harps=("HARPNUM", "nunique"))
mon["share_flagged"] = (mon.quality_nonzero / mon.records).round(3)
display(mon)
print("replay records", len(RP), "| quality_nonzero", int(RP.quality_nonzero.sum()), "| HARPs", RP.HARPNUM.nunique(),
      "| QUALITY values (hex, count):", {hex(int(k)): int(v) for k, v in RP.QUALITY.value_counts().head(6).items()})